# Document extraction cascade

**Recipe 39** · Level 3 (Intermediate) · Decision type: `Choice` + `Noul`

Select candidate values for document fields, verify each selection against its source span, and route uncertain or inconsistent records to review.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S06: [TypeSafe AI: Cookbooks](https://docs.typesafe.ai/cookbooks)

## What you will build

A two-stage cascade over a short purchase document with two fields, `invoice_number` and `total_amount`. Stage 1 is a `Choice` per field: which candidate span, if any, states that field's value, drawn from a fixed set Python extracts from the document text, plus a shared `not_found` fallback. Stage 2 is a `Noul` per field: does the span stage 1 picked really state that field, as opposed to a related but different figure (a purchase order number, a subtotal, a previous balance)? By the end you will have seen both typed answers for one document, the rule that composes them into one field result and then one document result, and an evaluation at both levels: per field, and per document.


## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays stored answers offline; setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls and changes nothing else. The header states which mode ran, and every number below carries that mode with it. `get_backend` is called exactly once, here, for the whole notebook: every request below reuses this one `backend`.


In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    evaluate_outcomes,
    noul_confidence,
    select_confidence_threshold,
    select_threshold,
    selective_curve,
    threshold_sweep,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_risk_coverage,
    plot_threshold_sweep,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path())  # the one backend this notebook uses
# scored excludes the two demo examples: they carry no gold label, so no metric below is
# about them. Each scored example is one document; each document has two fields, so the
# field-level metrics later cover twice as many (document, field) pairs as there are
# documents here -- examples and units are different quantities, both reported below.
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result. This label does not depend on the run mode, unlike `check` above.
selection = " (a selection step, not a reported result)"

run_header(
    39,
    "Document extraction cascade",
    backend=backend,
    n_examples=len(scored),
)

## The state

Python extracts the candidate spans before Jev ever sees the document: `helpers.extract_candidates` splits the document into sentences and keeps the ones that look like they could name a field (a reference-shaped number for `invoice_number`, a dollar amount for `total_amount`). The state for a stage-1 request is exactly those candidate spans, by id, for the fields that have at least one -- never the full document, and never an offset Jev would have to resolve itself.


In [ ]:
import json

demo = {e.id: e for e in examples if e.split == "demo"}
document = demo["d01-both-fields"]
print("Python has:")
print(json.dumps(document.fields, indent=2))
s1_fields = helpers.stage1_fields(document.fields)
state1 = helpers.build_stage1_state(document.fields, s1_fields)
print("Jev sees, in the stage-1 request:")
print(json.dumps(state1, indent=2))

## The questions

Stage 1 asks one `Choice` per field that has a candidate: which span, if any, states this field, with the shared `not_found` option added by Python for a document where none of them does. The two fields' `Choice` questions never depend on each other, so they travel together in one request.

Stage 2 asks one `Noul` per field stage 1 named a real candidate for: a statement that can be true or false, "the selected text really states this field". A field's own `Noul` instructions are fixed text; the candidate text being verified lives in the *state*, not the instructions, so the question itself never changes from one document to the next -- only what it is asked about does. A field stage 1 resolved to `not_found` is never verified at all: there is no text to check.


In [ ]:
questions1 = helpers.build_stage1_questions(document.fields, s1_fields)
for name, question in questions1.items():
    print(name, question.to_dict())

### The request boundary

A question that depends on an earlier answer goes in a later request; independent questions share one. Stage 2's own questions are built from stage 1's answers (`helpers.stage2_fields`, `helpers.build_stage2_state`), so stage 2 can never be sent before stage 1 returns -- but the two fields' stage-2 propositions still do not depend on *each other*, so whenever more than one field reaches stage 2, both travel together in the same, later, request. `helpers.plan_requests` makes this boundary a value the notebook can print, not only a rule to take on faith: it names which fields are bundled into which request, and which, if any, request it depended on.

Three documents below show the three shapes this takes: `d01-both-fields` makes two requests, each bundling both fields; `d02-no-candidates` makes none, because neither field has a candidate at all; `t07` makes two requests but the second bundles only one field, because `invoice_number`'s own stage-1 answer named `not_found` (a decoy tracking number was present, but nothing stated the invoice number) while `total_amount`'s named a real candidate.


In [ ]:
for doc_id in ("d01-both-fields", "d02-no-candidates", "t07"):
    fields = next(e for e in examples if e.id == doc_id).fields
    s1, a1, s2, a2 = helpers.decide_document(backend, fields)
    plan = helpers.plan_requests(s1, s2)
    print(f"{doc_id}:")
    if not plan:
        print("  no request: neither field has a candidate")
    for request in plan:
        depends = (
            f", depends on stage-{request.stage - 1}'s answer for {request.depends_on}"
            if request.depends_on
            else ""
        )
        print(f"  request (stage {request.stage}): fields {request.fields}{depends}")

## One answer up close

`d01-both-fields` has a real candidate for each field, so it makes both requests. The stage-1 `Choice` answer names a span and carries a `confidence` for it; the stage-2 `Noul` answer is a single probability that the selection really states the field, with no confidence of its own -- `noul_confidence` (`|2p - 1|`) derives one, usable the same way a `Choice` or `Score` confidence is.


In [ ]:
s1, stage1_answers, s2, stage2_answers = helpers.decide_document(backend, document.fields)
for field in helpers.FIELDS:
    print(f"-- {field} --")
    show_answer(stage1_answers[field])
    if field in stage2_answers:
        show_answer(stage2_answers[field])
        print(f"  noul_confidence: {noul_confidence([stage2_answers[field].noul])[0]:.2f}")

In [ ]:
plot_answer_probabilities(
    stage1_answers["invoice_number"], title="Stage 1: invoice_number candidate"
)

In [ ]:
plot_answer_probabilities(
    stage2_answers["invoice_number"], title="Stage 2: invoice_number verification"
)

## Python's part

Three checks, in this order, decide one field's result once stage 2 has answered: a defensive check that the choice really names one of this document's own candidates; then the model's own uncertainty (`noul_confidence` below the frozen confidence gate sends the field to [review](../../docs/glossary.md#review) as **uncertain**); then the verification itself (`noul` below the frozen [business threshold](../../docs/glossary.md#business-threshold) sends it to review as **inconsistent**). A field whose stage-1 answer is `not_found` is reported as `not_found` directly, whatever its confidence -- it is a complete answer, not a deferral, records no action, and being wrong leaves nothing standing beyond the missed field itself (`CONTRIBUTING.md` section 4's three-part test). A field with no candidate at all is resolved the same way, before any question is ever built.

The document is composed from its own two fields: `accepted` only when neither field was sent to review. Applying this to both documents above shows both outcomes: `d01-both-fields` has nothing to review and so is accepted; `d02-no-candidates` resolves both fields to `not_found` directly, with zero requests, and is accepted too, since `not_found` is itself a complete answer when nothing supports the field. A document in `fixtures/` with a real field in review (`v04`, `v08`, `t12`, ...) is composed as not accepted.


In [ ]:
# The two settings this rule needs (the confidence gate and the business threshold) are
# chosen on validation in "Evaluation" below; frozen here with their already-chosen values so
# this section can show the rule before the selection story. (The two numbers are not
# hand-picked: they are exactly what the Evaluation section's own selection cells compute.)
demo_business_threshold = 0.85
demo_confidence_gate = 0.70

for doc_id in ("d01-both-fields", "d02-no-candidates"):
    fields = next(e for e in examples if e.id == doc_id).fields
    s1, a1, s2, a2 = helpers.decide_document(backend, fields)
    results = {
        field: helpers.field_result(
            doc_id, field, fields, a1, a2, demo_business_threshold, demo_confidence_gate
        )
        for field in helpers.FIELDS
    }
    for field, result in results.items():
        print(f"{doc_id}/{field}: {result.outcome} ({result.reason})")
    outcome = helpers.compose_document_outcome(doc_id, results)
    print(f"{doc_id}: accepted={outcome.accepted}")

### Why both fields, and both stages, matter

Flipping one field's stage-1 choice, or one field's stage-2 verification, on a real document already in `fixtures/`, changes what the rule reports -- neither field and neither stage is along for the ride. `v11` has three candidate spans for `invoice_number` (two decoys, one real); naming a decoy instead of the real one, with its own stage-2 verification left exactly as it is, turns a correct field into a wrong one. `v01` is fully accepted as committed; flipping only its `invoice_number` verification from confidently-true to confidently-false, with its stage-1 choice left exactly as it is, sends that field to review and withholds the whole document.


In [ ]:
v11_fields = next(e for e in examples if e.id == "v11").fields
v11_spans = helpers.spans_by_id(v11_fields, "invoice_number")
real_choice = next(sid for sid, text in v11_spans.items() if "5126" in text)
decoy_choice = next(sid for sid in v11_spans if sid != real_choice)
real = helpers.finalize_field(
    "v11", "invoice_number", real_choice, v11_spans, 0.85, 0.70, 0.85, 0.70
)
decoy = helpers.finalize_field(
    "v11", "invoice_number", decoy_choice, v11_spans, 0.85, 0.70, 0.85, 0.70
)
print(
    f"v11/invoice_number, real choice:  outcome={real.outcome}, correct={helpers.matches_gold(real, '5126')}"
)
print(
    f"v11/invoice_number, decoy choice: outcome={decoy.outcome}, correct={helpers.matches_gold(decoy, '5126')}"
)

v01_fields = next(e for e in examples if e.id == "v01").fields
v01_spans = helpers.spans_by_id(v01_fields, "invoice_number")
v01_choice = next(iter(v01_spans))
amount_spans = helpers.spans_by_id(v01_fields, "total_amount")
amount = helpers.finalize_field(
    "v01", "total_amount", next(iter(amount_spans)), amount_spans, 0.90, 0.80, 0.85, 0.70
)
verified = helpers.finalize_field(
    "v01", "invoice_number", v01_choice, v01_spans, 0.90, 0.80, 0.85, 0.70
)
not_verified = helpers.finalize_field(
    "v01", "invoice_number", v01_choice, v01_spans, 0.10, 0.80, 0.85, 0.70
)
before = helpers.compose_document_outcome(
    "v01", {"invoice_number": verified, "total_amount": amount}
)
after = helpers.compose_document_outcome(
    "v01", {"invoice_number": not_verified, "total_amount": amount}
)
print(f"v01, invoice_number verified:     document accepted={before.accepted}")
print(f"v01, invoice_number not verified: document accepted={after.accepted}")

## Evaluation

Two settings are chosen on `validation` and then frozen: a [business threshold](../../docs/glossary.md#business-threshold) on the verification `noul` (the cut-off above which a verification counts as "yes, this really is the field"), and a confidence gate on `noul_confidence` (the cut-off below which even a "yes" or "no" is too unsure to trust automatically). Both are chosen from the (document, field) pairs where stage 2 actually asked a question -- a field resolved by `not_found` or by having no candidate at all never reaches stage 2, so it contributes no verification to choose from. "Truth" for this selection is whether the stage-1 choice was itself right, read from the [gold label](../../docs/glossary.md#gold-label) (`fixtures/labels.jsonl`, never shown to Jev).

Once frozen, both settings apply to every document, `validation` and `test` alike. The rule reports a field's result as `not_found`, `extracted`, or sent to [review](../../docs/glossary.md#review) for one of two reasons: **uncertain** (confidence below the gate) or **inconsistent** (verification below the threshold). [Coverage](../../docs/glossary.md#coverage) counts fields actually answered (`not_found` or `extracted`, never `review`); [risk](../../docs/glossary.md#risk) is the error rate among exactly those. This is reported at two levels: **field-level**, one row per (document, field) pair (the acceptance criteria's "field-level accuracy, review rate, and error rate among records not sent to review"); and **task-level**, one row per document, where a document is accepted only when neither of its fields was sent to review, and correct only when every field it reports is correct. In an offline run every number below is a pipeline check, and a `validation` number is additionally a selection step, never a reported result.


In [ ]:
def provisional_pairs(split_examples):
    """(truth, noul) for every (document, field) pair whose stage-1 choice named a real
    candidate -- exactly the pairs stage 2 asks a question about."""
    truth, noul = [], []
    for example in split_examples:
        s1, a1, s2, a2 = helpers.decide_document(backend, example.fields)
        gold = labels[example.id]
        for field in s2:
            spans = helpers.spans_by_id(example.fields, field)
            is_right = gold[field] != helpers.NOT_FOUND and gold[field] in spans[a1[field].choice]
            truth.append(is_right)
            noul.append(a2[field].noul)
    return truth, noul


val_examples = select_split(scored, "validation")
val_truth, val_noul = provisional_pairs(val_examples)
sweep = threshold_sweep(val_truth, val_noul)
business_threshold = select_threshold(val_truth, val_noul, "f1")
print(f"validation, {len(val_truth)} provisional (document, field) pairs{selection}{check}")
for point in sweep:
    print(
        f"  validation threshold {point.threshold:.2f}: precision {point.precision:.4f} "
        f"recall {point.recall:.4f} f1 {point.f1:.4f}{selection}{check}"
    )
print(
    f"chosen business threshold (max f1 on validation): {business_threshold:.4f}{selection}{check}"
)
plot_threshold_sweep(
    sweep, chosen=business_threshold, title=f"Validation verification sweep{check}"
)

In [ ]:
val_correct = [
    (n >= business_threshold) == truth for n, truth in zip(val_noul, val_truth, strict=True)
]
val_confidence = noul_confidence(val_noul)
# min_coverage, not target_accuracy=1.0: this pool's one irreducible verification mistake
# (v12/total_amount) has HIGH confidence (it is confidently wrong, not unsure), so no
# confidence gate can exclude it without excluding almost everything else too -- a
# target_accuracy of 1.0 would simply fail here, which is itself worth seeing rather than
# avoiding by construction.
confidence_gate = select_confidence_threshold(val_correct, val_confidence, min_coverage=0.85)
print(
    f"validation verification accuracy at every confidence: "
    f"{sum(val_correct)} of {len(val_correct)} right regardless of gate{selection}{check}"
)
print(
    f"chosen confidence gate (min_coverage=0.85 on validation): {confidence_gate:.4f}{selection}{check}"
)

### Field-level evaluation


In [ ]:
def field_results_for(split_examples):
    results = {}
    for example in split_examples:
        s1, a1, s2, a2 = helpers.decide_document(backend, example.fields)
        results[example.id] = {
            field: helpers.field_result(
                example.id, field, example.fields, a1, a2, business_threshold, confidence_gate
            )
            for field in helpers.FIELDS
        }
    return results


val_results = field_results_for(val_examples)
val_gold = {e.id: labels[e.id] for e in val_examples}
val_keys, val_accepted, val_field_correct = helpers.flatten_fields(
    [e.id for e in val_examples], val_results, val_gold
)
val_field_report = evaluate_outcomes(val_accepted, val_field_correct)
print(
    f"validation fields, n={val_field_report.n_total}: "
    f"coverage {val_field_report.coverage:.4f}, accuracy {val_field_report.accuracy:.4f}, "
    f"risk {val_field_report.risk:.4f}{selection}{check}"
)

In [ ]:
test_examples = select_split(scored, "test")
test_results = field_results_for(test_examples)
test_gold = {e.id: labels[e.id] for e in test_examples}
test_keys, test_accepted, test_field_correct = helpers.flatten_fields(
    [e.id for e in test_examples], test_results, test_gold
)
test_field_report = evaluate_outcomes(test_accepted, test_field_correct)
print(
    f"test fields, n={test_field_report.n_total}: "
    f"coverage {test_field_report.coverage:.4f}, accuracy {test_field_report.accuracy:.4f}, "
    f"risk {test_field_report.risk:.4f}{check}"
)
reasons = {}
for doc_id, field in test_keys:
    result = test_results[doc_id][field]
    if result.outcome == helpers.REVIEW:
        reasons[result.reason] = reasons.get(result.reason, 0) + 1
print(f"test fields sent to review, by reason{check}:")
for reason, count in sorted(reasons.items()):
    print(f"  {reason}: {count}{check}")

### Task-level evaluation

A document's own outcome composes its two fields: [accepted](../../docs/glossary.md#selective-prediction) only when neither field was sent to review, correct (among accepted documents) only when every field it reports matches its gold value.


In [ ]:
val_doc_accepted = [
    helpers.compose_document_outcome(e.id, val_results[e.id]).accepted for e in val_examples
]
val_doc_correct = [
    helpers.document_correct(val_results[e.id], val_gold[e.id]) if a else False
    for e, a in zip(val_examples, val_doc_accepted, strict=True)
]
val_task_report = evaluate_outcomes(val_doc_accepted, val_doc_correct)
print(
    f"validation documents, n={val_task_report.n_total}: "
    f"coverage {val_task_report.coverage:.4f}, accuracy {val_task_report.accuracy:.4f}, "
    f"risk {val_task_report.risk:.4f}{selection}{check}"
)

In [ ]:
test_doc_accepted = [
    helpers.compose_document_outcome(e.id, test_results[e.id]).accepted for e in test_examples
]
test_doc_correct = [
    helpers.document_correct(test_results[e.id], test_gold[e.id]) if a else False
    for e, a in zip(test_examples, test_doc_accepted, strict=True)
]
test_task_report = evaluate_outcomes(test_doc_accepted, test_doc_correct)
print(
    f"test documents, n={test_task_report.n_total}: "
    f"coverage {test_task_report.coverage:.4f}, accuracy {test_task_report.accuracy:.4f}, "
    f"risk {test_task_report.risk:.4f}{check}"
)

In [ ]:
test_truth, test_noul = provisional_pairs(test_examples)
test_correct_verif = [
    (n >= business_threshold) == truth for n, truth in zip(test_noul, test_truth, strict=True)
]
test_confidence = noul_confidence(test_noul)
curve = selective_curve(test_correct_verif, test_confidence)
print(f"test verification confidence sweep, {len(test_truth)} pairs{check}:")
for i in range(len(curve.thresholds)):
    print(
        f"  gate {curve.thresholds[i]:.2f}: coverage {curve.coverage[i]:.4f}, "
        f"accuracy {curve.accuracy[i]:.4f}, risk {curve.risk[i]:.4f}{check}"
    )
plot_risk_coverage(curve, title=f"Test verification confidence sweep{check}")

In [ ]:
print("Test fields the rule accepted but got wrong:")
for doc_id, field in test_keys:
    result = test_results[doc_id][field]
    gold_value = test_gold[doc_id][field]
    if result.outcome != helpers.REVIEW and not helpers.matches_gold(result, gold_value):
        print(
            f"  {doc_id}/{field}: outcome={result.outcome}, value={result.value!r}, "
            f"gold={gold_value!r}{check}"
        )

The cell above is why the rule is not enough on its own: `t04`'s `invoice_number` names a decoy ("Previous invoice reference 5998 was paid in full last month") at stage 1, and stage 2's own verification confidently, wrongly agrees -- confident enough to clear both the business threshold and the confidence gate. A verification stage can be fooled by the same kind of convincing decoy its stage-1 counterpart can, and no threshold chosen from this fixture pool catches every such case; the fixtures include this on purpose, and a recipe's evaluation should find it rather than average it away.


## What was and was not measured

What this run can say about Jev depends on its mode, which the cell below states from `backend.mode`: a synthetic or scripted run is a pipeline check only, on the fixture sample it ran against; a recorded or live run states the model the API returned and when, for exactly the examples it covers. Nothing here says anything about Jev's quality, speed or cost unless that cell reports a recorded or live run. `N` below counts the scored documents; the field-level and task-level evaluations above cover all of them, just at two different granularities.


In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test documents{check}")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test documents")

## Next steps

- Change the wording of `_STAGE1_INSTRUCTIONS["total_amount"]` in `helpers.py` and run `pytest recipes/39-document-extraction-cascade` or this notebook: the stored keys no longer match the question, so replay misses instead of guessing. `python build_fixtures.py` writes new keys but rewrites the committed fixtures; `git checkout recipes/39-document-extraction-cascade` restores them.
- Add a third field (a `due_date`, say) with its own extractor in `helpers.py`, and watch both the stage-1 and stage-2 requests grow by one question each, without the number of *requests* changing: the composition in `helpers.py` is written in terms of `FIELDS`, not a fixed pair.
- Neighbouring recipes: [`14-source-span-selection`](../14-source-span-selection/), whose single-stage `Choice`-over-spans pattern this recipe's own stage 1 reuses, and [`29-tool-outcome-verification`](../29-tool-outcome-verification/), another recipe whose second stage verifies what an earlier one produced.

A neighbour link above is a folder link, `../NN-slug/`: `tools/render_catalog.py` builds the root README's catalog table from bare recipe titles with no per-row anchor, so a recipe cannot reliably link to another recipe's row there instead. Both neighbours above are already published on `main`; a forward link to an unpublished recipe is not allowed ([docs/recipe-template.md](../../docs/recipe-template.md)).
